# Day 1 · Lab 1 — LangGraph Loan Eligibility Agent

## What you'll build

A stateful loan eligibility agent that:

1. Runs a LangGraph state machine over a `LoanState` schema
2. Routes conditionally based on eligibility score
3. Persists checkpoints to PostgreSQL (survives process restarts)
4. Pauses at a human-review gate via `interrupt_before`
5. Resumes after a human injects a decision via `update_state`
6. Traces every step to LangSmith automatically

## Sandbox facts (as-is)

- Working directory: `~/agentic-lab`
- Python: `/opt/miniconda/bin/python` (conda `(base)` env, Python 3.12.8) — **no venv needed**
- All packages preinstalled in `(base)`: `langgraph`, `langchain_openai`, `pydantic_ai`, `claude_agent_sdk`, `agents`, `mcp`
- `~/agentic-lab/.env` already contains: `DATABASE_URL`, `LANGSMITH_TRACING=true`, empty placeholders for `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `LANGSMITH_API_KEY`
- `OPENROUTER_API_KEY` is set at shell level by sandbox provisioning
- PostgreSQL `agentic-postgres` container running on port 5432

## In VS Code

`Ctrl+Shift+P` → `Python: Select Interpreter` → **`/opt/miniconda/bin/python`** (the `(base)` interpreter). Reload the notebook after.


## Step 1 — Environment diagnostic

Loads `~/agentic-lab/.env`, verifies required vars, checks Docker. Fix any ✗ before continuing.

Uses `override=False` so shell-level `OPENROUTER_API_KEY` is not clobbered by an empty `.env` entry.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# --- Load .env from sandbox root; DO NOT override existing shell env ---
try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

env_path = Path("~/agentic-lab/.env").expanduser()
if env_path.exists():
    load_dotenv(env_path, override=False)   # ← key: keep shell values
    print(f"✓ .env loaded from {env_path}")
else:
    print(f"✗ .env not found at {env_path}")

# Drop empty-string values that dotenv may have injected — they hide real shell values
for k in list(os.environ.keys()):
    if k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY","OPENROUTER_API_KEY","DATABASE_URL") and os.environ.get(k) == "":
        del os.environ[k]

print(f"\nPython: {sys.executable}")
print(f"Working dir: {os.getcwd()}\n")

required = {
    "OPENROUTER_API_KEY": "LLM gateway key (set by sandbox at shell level)",
    "DATABASE_URL":       "PostgreSQL connection string for checkpoints",
}
optional = {
    "LANGSMITH_TRACING":  "'true' enables LangSmith tracing",
    "LANGSMITH_API_KEY":  "Needed only if LANGSMITH_TRACING=true",
    "LANGSMITH_PROJECT":  "LangSmith project name (defaults to 'agentic-lab')",
}

all_ok = True
print("REQUIRED:")
for k, desc in required.items():
    v = os.environ.get(k)
    print(f"  {'✓' if v else '✗'} {k}  ({desc})")
    if not v: all_ok = False

print("\nOPTIONAL:")
for k, desc in optional.items():
    v = os.environ.get(k)
    marker = "✓" if v else "·"
    val = v if v else "(not set)"
    if k == "LANGSMITH_API_KEY" and v:
        val = f"{v[:8]}…"
    print(f"  {marker} {k}={val}  ({desc})")

# LangSmith default project if tracing on but no explicit project
if os.environ.get("LANGSMITH_TRACING","").lower() == "true" and not os.environ.get("LANGSMITH_PROJECT"):
    os.environ["LANGSMITH_PROJECT"] = "agentic-lab"
    print("  → LANGSMITH_PROJECT defaulted to 'agentic-lab'")

# Silence LangSmith warnings if tracing is on but key is missing
if os.environ.get("LANGSMITH_TRACING","").lower() == "true" and not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("  → LANGSMITH_TRACING flipped to false (no key present) to silence noise")

# Export OpenAI-compatible aliases (langchain-openai reads these)
if os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    print("\n✓ OpenAI-compatible aliases exported (OPENAI_API_KEY, OPENAI_BASE_URL)")

# Check Postgres container
print("\nDocker Postgres:")
try:
    r = subprocess.run(
        ["docker","ps","--filter","name=agentic-postgres","--format","{{.Names}}\t{{.Status}}"],
        capture_output=True, text=True, timeout=5,
    )
    if r.stdout.strip():
        print(f"  ✓ {r.stdout.strip()}")
    else:
        print("  ✗ agentic-postgres not running. Fix: cd ~/agentic-lab && docker compose up -d")
        all_ok = False
except Exception as e:
    print(f"  ✗ docker check failed: {e}")

print("\n" + ("✓ All checks passed. Continue." if all_ok else "⚠  Fix ✗ items above before continuing."))

## Step 2 — State schema

`LoanState` is the contract every node commits to. Change the schema after checkpoints exist and old rows become unreadable.

- Simple field (`application_id: str`): last write wins
- Annotated reducer (`messages: Annotated[list, operator.add]`): each node's list is appended

In [ ]:
from typing import TypedDict, Annotated
import operator


class LoanState(TypedDict):
    # Simple fields (last write wins)
    application_id: str
    applicant_name: str
    loan_amount: float
    monthly_income: float

    # Accumulative — every node's list is APPENDED
    messages: Annotated[list, operator.add]

    # Owned fields — one node writes each
    eligibility_score: float
    bureau_score: int
    decision: str

    # HITL field
    human_feedback: dict


print("LoanState schema defined.")

## Step 3 — Deterministic node: eligibility check

No LLM. Pure Python score from income vs. loan amount.

In [ ]:
def check_eligibility(state: LoanState) -> dict:
    """Score = min(100, (annual_income / loan_amount) * 25)."""
    income = state["monthly_income"]
    amount = state["loan_amount"]
    if amount <= 0:
        return {"eligibility_score": 0.0, "messages": ["eligibility: invalid amount"]}
    score = min(100.0, (income * 12 / amount) * 25)
    return {
        "eligibility_score": score,
        "messages": [f"eligibility_check: score={score:.1f}"],
    }


def route_by_score(state: LoanState) -> str:
    s = state["eligibility_score"]
    if s >= 80: return "high"
    if s >= 50: return "review"
    return "low"


# Smoke test
sample = {"monthly_income": 6000, "loan_amount": 200_000, "messages": []}
result = check_eligibility(sample)
print(result)
print("Route:", route_by_score({**sample, **result}))

## Step 4 — LLM node: bureau lookup

`ChatOpenAI` pointed at OpenRouter with a provider-prefixed model ID: `anthropic/claude-sonnet-4.5`.

In production this would call a real bureau API. Here the LLM simulates a bureau response as structured JSON.

In [ ]:
from langchain_openai import ChatOpenAI
import json

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


def bureau_lookup(state: LoanState) -> dict:
    """Simulate a bureau lookup with an LLM. Return score 300-850."""
    prompt = (
        f"Given an applicant with monthly income {state['monthly_income']:.0f} "
        f"and loan amount {state['loan_amount']:.0f}, return a plausible credit bureau "
        f"response as JSON with keys 'score' (int 300-850) and 'tier' (low/medium/high). "
        f"Respond ONLY with the JSON object, no prose."
    )
    text = llm.invoke(prompt).content.strip()
    # Strip markdown fences if the model wraps in ```
    if text.startswith("```"):
        text = text.strip("`").split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        if text.startswith("json"):
            text = text[4:].strip()
    try:
        score = int(json.loads(text)["score"])
    except Exception:
        score = 700   # safe fallback for demo
    return {
        "bureau_score": score,
        "messages": [f"bureau_lookup: score={score}"],
    }


# Smoke test — actual call through OpenRouter
print("Calling OpenRouter (anthropic/claude-sonnet-4.5)...")
sample_state = {"monthly_income": 6000, "loan_amount": 200_000, "messages": []}
print(bureau_lookup(sample_state))

## Step 5 — Human review + final decision

`human_review` is a placeholder — real gating comes from `interrupt_before` on the graph. `make_decision` combines bureau score + human feedback.

In [ ]:
def human_review(state: LoanState) -> dict:
    """Placeholder — actual gating comes from interrupt_before."""
    return {"messages": [f"human_review: feedback={state.get('human_feedback', {})}"]}


def make_decision(state: LoanState) -> dict:
    """Combine bureau score + human feedback into a final decision."""
    bureau = state.get("bureau_score", 0)
    human = state.get("human_feedback", {})
    if human.get("approved") is True:
        decision = "approved"
    elif human.get("approved") is False:
        decision = "rejected"
    elif bureau >= 700:
        decision = "approved"
    elif bureau >= 600:
        decision = "review"
    else:
        decision = "rejected"
    return {
        "decision": decision,
        "messages": [f"decision: {decision} (bureau={bureau}, human={human})"],
    }


print("Node functions defined.")

## Step 6 — Assemble the graph

Edges:
- `START → eligibility`
- `eligibility → {high: bureau, review: human_review, low: END}`  (conditional)
- `bureau → decision`
- `human_review → decision`
- `decision → END`

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(LoanState)
builder.add_node("eligibility", check_eligibility)
builder.add_node("bureau", bureau_lookup)
builder.add_node("human_review", human_review)
builder.add_node("decision", make_decision)

builder.add_edge(START, "eligibility")
builder.add_conditional_edges(
    "eligibility",
    route_by_score,
    {"high": "bureau", "review": "human_review", "low": END},
)
builder.add_edge("bureau", "decision")
builder.add_edge("human_review", "decision")
builder.add_edge("decision", END)

print("Graph builder assembled with nodes:", list(builder.nodes))

## Step 7 — Compile with PostgresSaver + HITL interrupt

`PostgresSaver.from_conn_string(...)` is a **context manager**. `cp.setup()` on first run creates the checkpoint tables.

The interrupt is on **`compile()`**, not on `add_node`.

In [ ]:
from langgraph.checkpoint.postgres import PostgresSaver

DB_URL = os.environ["DATABASE_URL"]

# Enter the context here; store the checkpointer for reuse across cells.
# We close it in the final cleanup cell.
_pg_ctx = PostgresSaver.from_conn_string(DB_URL)
checkpointer = _pg_ctx.__enter__()
checkpointer.setup()   # idempotent — creates tables on first run

graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["human_review"],   # ← HITL gate
)

print("Graph compiled with PostgresSaver checkpointer and HITL on human_review.")

## Step 8 — Run until the HITL interrupt

Test values chosen so eligibility lands in the `review` band (50–79) — graph will pause at `human_review`.

In [ ]:
import uuid

app_id = f"APP-{uuid.uuid4().hex[:8].upper()}"
config = {"configurable": {"thread_id": f"loan-{app_id}"}}

initial_state = {
    "application_id": app_id,
    "applicant_name": "Test Applicant",
    "loan_amount": 200_000,
    "monthly_income": 4500,   # → ratio 0.27, score ~67 → 'review' band
    "messages": [],
    "eligibility_score": 0.0,
    "bureau_score": 0,
    "decision": "",
    "human_feedback": {},
}

print(f"Running graph for {app_id}...")
result = graph.invoke(initial_state, config)

print("\nGraph paused. Values so far:")
for k, v in result.items():
    if k != "messages":
        print(f"  {k}: {v}")
print(f"  messages: {len(result['messages'])} entries")

## Step 9 — Inspect where the graph paused

`graph.get_state()` returns a snapshot. `state.next` shows which node would have run next.

In [ ]:
state = graph.get_state(config)
print("Next node (would have run):", state.next)
print("Current values:")
for k, v in state.values.items():
    if k != "messages":
        print(f"  {k}: {v}")
print("\nMessages so far:")
for m in state.values["messages"]:
    print(" ", m)

## Step 10 — Inject the human decision

`update_state` writes into the frozen state as if a node had run and produced this.

In [ ]:
graph.update_state(config, {
    "human_feedback": {
        "approved": True,
        "reviewer": "trainer",
        "note": "Manual approval — income stable per employer verification",
    }
})

state = graph.get_state(config)
print("human_feedback injected:", state.values["human_feedback"])
print("Next node (still):", state.next)

## Step 11 — Resume with `graph.invoke(None, config)`

`None` = *continue from checkpoint*. Passing fresh state here would start a new run and lose the injected feedback.

In [ ]:
final = graph.invoke(None, config)

print("Graph resumed and completed.\n")
print("Final state:")
for k, v in final.items():
    if k != "messages":
        print(f"  {k}: {v}")
print("\nFull message log:")
for m in final["messages"]:
    print(" ", m)

## Step 12 — Verify persistence

The same `thread_id` reconstitutes state from PostgreSQL. Inspect the checkpoint history.

In [ ]:
history = list(graph.get_state_history(config))
print(f"Checkpoints for {config['configurable']['thread_id']}:")
print(f"  {len(history)} checkpoints written")
print("\nMost recent first:")
for i, snap in enumerate(history):
    nxt = snap.next if snap.next else "(complete)"
    print(f"  [{i}] next={nxt}")

print("\nInspect from psql:")
print("  docker exec -it agentic-postgres psql -U labuser -d agentdb")
print("  \\dt        # → checkpoints, checkpoint_blobs, checkpoint_writes tables")

## Step 13 — LangSmith trace (if enabled)

In [ ]:
project = os.environ.get("LANGSMITH_PROJECT", "(unset)")
tracing = os.environ.get("LANGSMITH_TRACING", "false")
print(f"LANGSMITH_TRACING: {tracing}")
print(f"LANGSMITH_PROJECT: {project}")
if tracing.lower() == "true":
    print("\n→ https://smith.langchain.com")
    print(f"  Project: {project}")
    print(f"  Filter by thread_id: loan-{app_id}")
else:
    print("\nTracing off. Add LANGSMITH_API_KEY to ~/agentic-lab/.env, re-run Step 1.")

## Cleanup — close the PostgresSaver context

In [ ]:
_pg_ctx.__exit__(None, None, None)
print("PostgresSaver closed. Notebook complete.")

## What you learned

1. **State schema is a contract** — TypedDict + Annotated reducers
2. **Nodes are pure functions** — take state, return partial state
3. **Conditional routing** — routing fn returns a node name string
4. **PostgreSQL checkpointing** — survives restarts, thread-isolated
5. **HITL pattern** — `interrupt_before` + `get_state` + `update_state` + `invoke(None)`
6. **OpenRouter integration** — `ChatOpenAI` with `base_url` + provider-prefixed model ID
7. **LangSmith automatic tracing** — zero code, env vars in `.env`

## Stretch

Open `lab2_sdk_comparison.ipynb` for the Claude Agent SDK + OpenAI Agents SDK versions.
